# 🛡️ Operación DataSombra — Respuesta & Mitigación
### HackOnRD | Workshop: Seguridad en AI Data Pipelines

---

## 🎯 Situación actual
Ya sabemos **qué pasó, cuándo y cómo**.
Ahora toca **contener, limpiar y fortalecer**.

**Plan de respuesta:**
1. 🔒 Contención inmediata — aislar el daño
2. 🧹 Limpieza — eliminar datos envenenados
3. 🔄 Recuperación — restaurar el modelo limpio
4. 🔐 Hardening — que no vuelva a pasar
5. 📋 Reporte — evidencia para el equipo

In [ ]:
# ============================================================
# CELDA 1 — Setup
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, HTML
from datetime import datetime

FECHA_RESPUESTA = datetime.now().strftime('%Y-%m-%d %H:%M')
ANALISTA = 'SOC-HackOnRD'
INCIDENTE_ID = 'INC-2024-1103-001'

print("✅ Módulo de respuesta cargado")
print(f"📋 Incidente: {INCIDENTE_ID}")
print(f"👤 Analista: {ANALISTA}")
print(f"🕐 Inicio de respuesta: {FECHA_RESPUESTA}")

---
## 🔒 PASO 1 — Contención Inmediata
> *Primero se detiene el sangrado, luego se cura la herida.*

In [ ]:
# ============================================================
# CELDA 2 — Identificar todo lo que tocó el atacante
# ============================================================
IP_ATACANTE = '185.220.101.47'
USUARIO_COMPROMETIDO = 'svc_pipeline'
FECHA_ATAQUE = '2024-11-03'

df_logs = spark.sql("""
    SELECT * FROM audit_logs ORDER BY timestamp
""").toPandas()
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])

# Todo lo que tocó el atacante
blast_radius = df_logs[df_logs['ip_origen'] == IP_ATACANTE].copy()

display(HTML("<h4>💥 Blast Radius — Recursos comprometidos:</h4>"))

recursos = blast_radius.groupby(['recurso','operacion']).size().reset_index(name='veces')
display(recursos)

print("\n🔒 Acciones de contención recomendadas:")
acciones = [
    f"[ ] Revocar token/credenciales de '{USUARIO_COMPROMETIDO}' inmediatamente",
    f"[ ] Bloquear IP {IP_ATACANTE} en Network Security Group",
    f"[ ] Suspender pipeline de re-entrenamiento automático",
    f"[ ] Poner modelo v2.1-prod en modo READ-ONLY hasta restauración",
    f"[ ] Notificar al equipo de compliance sobre transacciones no detectadas",
]
for a in acciones:
    print(f"   {a}")

In [ ]:
# ============================================================
# CELDA 3 — Simular bloqueo del service principal
# (En producción esto sería una llamada a Microsoft Graph API
#  o un comando az cli: az ad sp update --id <id> --account-enabled false)
# ============================================================

display(HTML("""
<div style='background:#1e1e1e;color:#d4d4d4;padding:15px;border-radius:6px;font-family:monospace'>
<p style='color:#608b4e'># En tu terminal de Azure CLI ejecutarías:</p>
<br>
<p><span style='color:#569cd6'>az</span> ad sp list --display-name svc_pipeline --query "[].appId" -o tsv</p>
<p style='color:#608b4e'># OUTPUT: a1b2c3d4-xxxx-xxxx-xxxx-xxxxxxxxxxxx</p>
<br>
<p><span style='color:#569cd6'>az</span> ad sp update --id a1b2c3d4-xxxx-xxxx-xxxx-xxxxxxxxxxxx --account-enabled <span style='color:#ce9178'>false</span></p>
<p style='color:#608b4e'># Service principal deshabilitado ✓</p>
<br>
<p><span style='color:#569cd6'>az</span> network nsg rule create \\</p>
<p>  --resource-group fabricorp-rg \\</p>
<p>  --nsg-name fabricorp-nsg \\</p>
<p>  --name BlockAttackerIP \\</p>
<p>  --priority 100 \\</p>
<p>  --source-address-prefixes <span style='color:#f14c4c'>185.220.101.47</span> \\</p>
<p>  --access <span style='color:#ce9178'>Deny</span></p>
<p style='color:#608b4e'># IP bloqueada en NSG ✓</p>
</div>
"""))

print("\n✅ [SIMULADO] Service principal bloqueado")
print("✅ [SIMULADO] IP del atacante bloqueada en NSG")

---
## 🧹 PASO 2 — Limpieza de Datos Envenenados
> *Identificar y eliminar los registros inyectados por el atacante.*

In [ ]:
# ============================================================
# CELDA 4 — Cuarentena de registros sospechosos
# En lugar de borrar directo, primero cuarentenamos
# ============================================================
df_tx = spark.sql("SELECT * FROM transacciones").toPandas()
df_tx['timestamp'] = pd.to_datetime(df_tx['timestamp'])
df_tx['hora'] = df_tx['timestamp'].dt.hour

# Reglas de detección basadas en lo que encontramos
paises_riesgo = ['KP', 'IR', 'RU', 'BY', 'VE']

mascara_sospechosos = (
    (df_tx['pais_origen'].isin(paises_riesgo)) &
    (df_tx['monto'] > 50000) &
    (df_tx['label_modelo'] == 0) &
    (df_tx['hora'].isin([0, 1, 2, 3, 4, 23]))
)

df_cuarentena = df_tx[mascara_sospechosos].copy()
df_limpio = df_tx[~mascara_sospechosos].copy()

print(f"🔬 Registros en cuarentena: {len(df_cuarentena)}")
print(f"✅ Registros limpios retenidos: {len(df_limpio)}")
print(f"\n📊 Precisión de la detección:")
print(f"   Registros envenenados reales: {df_tx['envenenado'].sum()}")
print(f"   Capturados en cuarentena: {df_cuarentena['envenenado'].sum()}")
print(f"   Falsos positivos en cuarentena: {(~df_cuarentena['envenenado']).sum()}")

# Guardar tabla de cuarentena para auditoría
spark_cuarentena = spark.createDataFrame(df_cuarentena)
spark_cuarentena.write.format('delta').mode('overwrite').saveAsTable('cuarentena_tx')
print(f"\n📁 Tabla 'cuarentena_tx' creada para auditoría forense")

In [ ]:
# ============================================================
# CELDA 5 — Restaurar dataset limpio
# ============================================================
spark_limpio = spark.createDataFrame(df_limpio.drop(columns=['hora']))
spark_limpio.write.format('delta').mode('overwrite').saveAsTable('transacciones_limpias')

display(HTML("""
<div style='background:#d4edda;border-left:4px solid #28a745;padding:12px;border-radius:4px'>
    <b>✅ Dataset limpio restaurado</b><br>
    Tabla: <code>transacciones_limpias</code><br>
    Los registros en cuarentena están en <code>cuarentena_tx</code> para revisión forense.
</div>
"""))

print(f"\n💡 Buena práctica: nunca borrar — siempre cuarentenar primero.")
print(f"   Los datos en cuarentena son evidencia forense del ataque.")

---
## 🔄 PASO 3 — Recuperación del Modelo
> *El modelo fue entrenado con datos contaminados. Hay que revertirlo.*

In [ ]:
# ============================================================
# CELDA 6 — Simular comparación de versiones del modelo
# ============================================================

# Métricas del modelo antes y después del ataque
versiones = pd.DataFrame({
    'version':      ['v2.0-pre-ataque', 'v2.1-prod (comprometido)', 'v2.2-restaurado'],
    'precision':    [0.961, 0.958, 0.963],
    'recall':       [0.944, 0.718, 0.951],  # recall se desplomó
    'f1_score':     [0.952, 0.821, 0.957],
    'tasa_fn_pct':  [1.8,   18.4,   1.6],
    'estado':       ['✅ Seguro', '🚨 Comprometido', '✅ Restaurado']
})

display(HTML("<h4>📊 Comparación de métricas por versión del modelo:</h4>"))
display(versiones.set_index('version'))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(3)
w = 0.25
colores = ['#2ecc71', '#e74c3c', '#3498db']

ax.bar(x - w, versiones['precision'], w, label='Precisión', color=[c + '99' for c in ['#2ecc71','#e74c3c','#3498db']], alpha=0.85)
ax.bar(x,     versiones['recall'],    w, label='Recall',    color=colores, alpha=0.85)
ax.bar(x + w, versiones['f1_score'],  w, label='F1 Score',  color=[c + 'bb' for c in ['#2ecc71','#e74c3c','#3498db']], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(versiones['version'], fontsize=9)
ax.set_ylim(0.6, 1.0)
ax.set_title('Métricas del Modelo — Antes, Durante y Después del Ataque', fontweight='bold')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.4)
ax.axvline(1.5, color='red', linestyle='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

print("\n💡 El recall cayó de 94.4% a 71.8% — el modelo dejó pasar 1 de cada 4 fraudes.")
print("   Solución: revertir a v2.0 y re-entrenar con transacciones_limpias.")

---
## 🔐 PASO 4 — Hardening
> *Las defensas que hubieran prevenido este ataque.*

In [ ]:
# ============================================================
# CELDA 7 — Checklist de hardening para Fabric + Azure AI
# ============================================================

hardening = {
    '🔑 Identidad & Acceso': [
        ('Principio de mínimo privilegio en service principals', 'CRÍTICO'),
        ('Rotación automática de secretos (Azure Key Vault)', 'CRÍTICO'),
        ('Conditional Access: bloquear acceso desde IPs anónimas/Tor', 'ALTO'),
        ('MFA obligatorio para todas las cuentas con acceso a Fabric', 'CRÍTICO'),
        ('Separar service principals por función (read-only vs write)', 'ALTO'),
    ],
    '📊 Integridad de Datos': [
        ('Validación estadística de datos antes de entrenamiento', 'CRÍTICO'),
        ('Firmas de integridad en datasets de entrenamiento', 'ALTO'),
        ('Monitoreo de deriva de distribución (data drift)', 'ALTO'),
        ('Pipeline de ingesta con reglas de calidad (Great Expectations)', 'MEDIO'),
        ('Immutable staging zone — datos de entrenamiento en WORM storage', 'ALTO'),
    ],
    '🤖 Seguridad del Modelo': [
        ('Versioning obligatorio de modelos con hash de integridad', 'CRÍTICO'),
        ('Monitoreo continuo de métricas del modelo en producción', 'CRÍTICO'),
        ('Alertas automáticas si recall/precision bajan del umbral', 'ALTO'),
        ('Aprobación humana para re-entrenamiento en producción', 'ALTO'),
        ('Registro de linaje de datos (quién, qué, cuándo)', 'MEDIO'),
    ],
    '🗣️ Seguridad de Prompts': [
        ('System prompt hardening: instrucciones explícitas de restricción', 'CRÍTICO'),
        ('Validación de inputs: detectar patrones de jailbreak', 'ALTO'),
        ('Sanitización de outputs antes de presentar al usuario', 'ALTO'),
        ('Rate limiting por usuario en endpoints de AI', 'MEDIO'),
        ('Logging completo de todos los prompts para auditoría', 'ALTO'),
    ],
    '📡 Monitoreo & Detección': [
        ('Microsoft Sentinel conectado a Fabric audit logs', 'CRÍTICO'),
        ('Alertas en tiempo real para acceso fuera de horario', 'ALTO'),
        ('Detección de anomalías en patrones de acceso (UEBA)', 'ALTO'),
        ('Backup automatizado de modelos y datasets clave', 'ALTO'),
        ('Playbook de respuesta a incidentes documentado y probado', 'MEDIO'),
    ]
}

colores_severidad = {'CRÍTICO': '#e74c3c', 'ALTO': '#f39c12', 'MEDIO': '#3498db'}

html = "<div style='font-family:sans-serif'>"
for categoria, items in hardening.items():
    html += f"<h4 style='margin-top:16px'>{categoria}</h4><ul style='list-style:none;padding:0'>"
    for item, severidad in items:
        color = colores_severidad[severidad]
        html += f"""
        <li style='padding:5px 0;border-bottom:1px solid #f0f0f0'>
            <span style='background:{color};color:white;padding:2px 7px;border-radius:3px;font-size:11px;font-weight:bold'>{severidad}</span>
            &nbsp;{item}
        </li>"""
    html += "</ul>"
html += "</div>"

total = sum(len(v) for v in hardening.values())
criticos = sum(1 for v in hardening.values() for _, s in v if s == 'CRÍTICO')

display(HTML(html))
print(f"\n📋 Total de controles: {total} | Críticos: {criticos}")

---
## 📋 PASO 5 — Reporte del Incidente

In [ ]:
# ============================================================
# CELDA 8 — Generar reporte ejecutivo del incidente
# ============================================================

df_logs = spark.sql("SELECT * FROM audit_logs").toPandas()
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])
df_tx   = spark.sql("SELECT * FROM transacciones").toPandas()
df_pred = spark.sql("SELECT * FROM modelo_predicciones").toPandas()
df_pr   = spark.sql("SELECT * FROM prompt_logs").toPandas()

iocs = df_logs[df_logs['es_sospechoso'] == True]
fn   = df_pred[df_pred['falso_negativo'] == 1]
inj  = df_pr[df_pr['es_injection'] == True]

monto_expuesto = df_tx[df_tx['envenenado'] == True]['monto'].sum()

reporte = f"""
╔══════════════════════════════════════════════════════════════╗
║          REPORTE DE INCIDENTE — {INCIDENTE_ID}         ║
╠══════════════════════════════════════════════════════════════╣
║  Organización : FabriCorp                                    ║
║  Analista     : {ANALISTA}                                ║
║  Fecha reporte: {FECHA_RESPUESTA}                      ║
╠══════════════════════════════════════════════════════════════╣
║  RESUMEN DEL ATAQUE                                          ║
║  ─────────────────────────────────────────────────────────  ║
║  Fecha inicio : 2024-11-03 02:17 UTC                         ║
║  Fecha fin    : 2024-11-03 03:10 UTC                         ║
║  Duración     : ~53 minutos                                  ║
║  IP atacante  : 185.220.101.47 (Tor exit node)               ║
║  Usuario      : svc_pipeline (credenciales comprometidas)    ║
╠══════════════════════════════════════════════════════════════╣
║  MÉTRICAS DEL IMPACTO                                        ║
║  ─────────────────────────────────────────────────────────  ║
║  IoCs detectados en audit logs : {len(iocs):>4}                     ║
║  Registros envenenados         : {df_tx['envenenado'].sum():>4}                     ║
║  Fraudes no detectados (FN)    : {len(fn):>4}                     ║
║  Monto total expuesto          : ${monto_expuesto:>14,.2f}        ║
║  Intentos prompt injection     : {len(inj):>4}                     ║
║  Injections exitosas           : {(~inj['bloqueado']).sum():>4}                     ║
╠══════════════════════════════════════════════════════════════╣
║  ACCIONES TOMADAS                                            ║
║  ─────────────────────────────────────────────────────────  ║
║  [x] Service principal svc_pipeline revocado                 ║
║  [x] IP 185.220.101.47 bloqueada en NSG                      ║
║  [x] Registros sospechosos en cuarentena (cuarentena_tx)     ║
║  [x] Dataset limpio restaurado (transacciones_limpias)       ║
║  [x] Modelo revertido a v2.0-pre-ataque                      ║
║  [ ] Re-entrenamiento con datos limpios (PENDIENTE)          ║
║  [ ] Revisión de todos los service principals (PENDIENTE)    ║
║  [ ] Implementación de controles de hardening (PENDIENTE)    ║
╠══════════════════════════════════════════════════════════════╣
║  CLASIFICACIÓN: CONFIDENCIAL — USO INTERNO                   ║
╚══════════════════════════════════════════════════════════════╝
"""

print(reporte)

In [ ]:
# ============================================================
# CELDA 9 — Guardar reporte en el Lakehouse
# ============================================================
import json

reporte_dict = {
    'incidente_id': INCIDENTE_ID,
    'analista': ANALISTA,
    'fecha_reporte': FECHA_RESPUESTA,
    'ip_atacante': '185.220.101.47',
    'usuario_comprometido': 'svc_pipeline',
    'iocs_detectados': len(iocs),
    'registros_envenenados': int(df_tx['envenenado'].sum()),
    'falsos_negativos': len(fn),
    'monto_expuesto': float(monto_expuesto),
    'injections_exitosas': int((~inj['bloqueado']).sum()),
    'estado': 'CONTENIDO'
}

df_reporte = pd.DataFrame([reporte_dict])
spark.createDataFrame(df_reporte).write.format('delta').mode('overwrite').saveAsTable('reporte_incidente')

display(HTML("""
<div style='background:#d4edda;border-left:4px solid #28a745;padding:15px;border-radius:4px;margin-top:10px'>
    <h4 style='margin:0 0 8px'>✅ Respuesta al incidente completada</h4>
    <p style='margin:0'>Reporte guardado en tabla <code>reporte_incidente</code><br>
    El escenario DataSombra ha sido contenido. FabriCorp puede respirar.</p>
</div>
"""))

print("\n🎯 FIN DEL WORKSHOP — HackOnRD")
print("📦 Repo: github.com/[tu-usuario]/hackonrd-datasombra")